# PyTorch LSTM trénink — vylepšený behavior klasifikátor

Vylepšení oproti původnímu Keras `detekce_novy_pristub.ipynb`:
- **PyTorch** místo TF/Keras
- **Bidirectional LSTM** + **multi-head self-attention**
- **Skeleton normalizace**: centrování na hip midpoint, scale podle torso délky
- **Velocity features**: frame-to-frame deltas jako další vstup
- **Class weights + Focal loss** pro řešení class imbalance
- **Augmentace**: horizontal flip (s left/right swap), random rotation, scale, jitter
- **CrowdPose 14 keypoints** (current ViTPose output) — konvertuje legacy COCO-17 labely

**Datasety:**
1. `labeled_behaviors/` — manual labels z MED (4 sekvence, 196 framů, 3 třídy)
2. `data/RWF-2000/skeletons/` — RWF processed pickles (po spuštění process_rwf.ipynb)

**Class mapping:**
- MED: 0=walking/normal, 1=suspicious, 2=running/panic
- RWF binary → MED: NonFight (0) → walking (0), Fight (1) → suspicious (1)

In [3]:
# === Cell 0: Imports ===
import os
import glob
import json
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}, Device: {DEVICE}')

PyTorch: 2.1.0+cu121, Device: cuda


C:\Users\tomas\anaconda3\envs\ocv412cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# === Cell 1: Configuration ===
N_KEYPOINTS = 14         # CrowdPose (ViTPose output)
SEQUENCE_LENGTH = 15     # frames per LSTM input window (15 @ 5 FPS = 3 sec context)
STEP_SIZE = 5            # sliding window step

# 2-class binary: NonFight (=walking/normal) vs Fight (=suspicious)
CLASS_NAMES = ['NonFight', 'Fight']
N_CLASSES = len(CLASS_NAMES)

# Training hyperparameters
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0

# Loss
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
USE_AUGMENTATION = True

# Architecture
HIDDEN_DIM = 128
N_LSTM_LAYERS = 2
ATTENTION_HEADS = 4
DROPOUT = 0.3

# Paths (RWF only)
RWF_SKELETONS_DIR = 'data/RWF-2000/skeletons'
MODEL_SAVE_PATH = 'best_lstm_pytorch.pt'

# Random seed
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [5]:
# === Cell 2: Skeleton format conversion (COCO 17 → CrowdPose 14) ===
# COCO 17 layout:    0=nose, 1=l_eye, 2=r_eye, 3=l_ear, 4=r_ear,
#                    5=l_shoulder, 6=r_shoulder, 7=l_elbow, 8=r_elbow,
#                    9=l_wrist, 10=r_wrist, 11=l_hip, 12=r_hip,
#                    13=l_knee, 14=r_knee, 15=l_ankle, 16=r_ankle
# CrowdPose 14:      0=l_shoulder, 1=r_shoulder, 2=l_elbow, 3=r_elbow,
#                    4=l_wrist, 5=r_wrist, 6=l_hip, 7=r_hip,
#                    8=l_knee, 9=r_knee, 10=l_ankle, 11=r_ankle,
#                    12=head (avg of nose+eyes+ears), 13=neck (avg shoulders)

def coco17_to_crowdpose14(skeleton):
    """Convert (T, 17, D) COCO → (T, 14, D) CrowdPose. D=2 nebo 3."""
    if skeleton.shape[-2] == 14:
        return skeleton
    T = skeleton.shape[0]
    D = skeleton.shape[-1]
    out = np.zeros((T, 14, D), dtype=skeleton.dtype)

    # body keypoints - direct mapping
    out[:, 0]  = skeleton[:, 5]     # l_shoulder
    out[:, 1]  = skeleton[:, 6]     # r_shoulder
    out[:, 2]  = skeleton[:, 7]     # l_elbow
    out[:, 3]  = skeleton[:, 8]     # r_elbow
    out[:, 4]  = skeleton[:, 9]     # l_wrist
    out[:, 5]  = skeleton[:, 10]    # r_wrist
    out[:, 6]  = skeleton[:, 11]    # l_hip
    out[:, 7]  = skeleton[:, 12]    # r_hip
    out[:, 8]  = skeleton[:, 13]    # l_knee
    out[:, 9]  = skeleton[:, 14]    # r_knee
    out[:, 10] = skeleton[:, 15]    # l_ankle
    out[:, 11] = skeleton[:, 16]    # r_ankle
    out[:, 12] = skeleton[:, 0:5].mean(axis=1)              # head (avg head pts)
    out[:, 13] = (skeleton[:, 5] + skeleton[:, 6]) / 2      # neck (avg shoulders)
    return out


def coco17_flat_to_crowdpose14_xy(flat_34):
    """(T, 34) flat COCO 17 → (T, 14, 2) CrowdPose."""
    T = flat_34.shape[0]
    sk_17 = flat_34.reshape(T, 17, 2)
    return coco17_to_crowdpose14(sk_17)

In [6]:
# === Cell 3: Skeleton preprocessing (normalize + velocity features) ===
def normalize_skeleton(skeleton, eps=1e-6):
    """
    Center on hip midpoint, scale by torso (shoulder-hip) length.
    Handles missing keypoints (zeros) gracefully.

    Args:
        skeleton: (T, K=14, D) where D >= 2

    Returns:
        normalized (T, K, D), only first 2 dims are modified
    """
    sk = skeleton.copy()
    # CrowdPose indices
    L_SHOULDER, R_SHOULDER = 0, 1
    L_HIP, R_HIP = 6, 7

    hip_l = sk[:, L_HIP, :2]                                # (T, 2)
    hip_r = sk[:, R_HIP, :2]
    shoulder_l = sk[:, L_SHOULDER, :2]
    shoulder_r = sk[:, R_SHOULDER, :2]

    hip_mid = (hip_l + hip_r) / 2                            # (T, 2)
    shoulder_mid = (shoulder_l + shoulder_r) / 2

    torso = np.linalg.norm(shoulder_mid - hip_mid, axis=-1, keepdims=True)  # (T, 1)
    torso = np.maximum(torso, eps)

    # Center
    sk[..., :2] = sk[..., :2] - hip_mid[:, None, :]
    # Scale
    sk[..., :2] = sk[..., :2] / torso[:, :, None]
    return sk


def add_velocity_features(skeleton):
    """
    Append velocity (frame-to-frame deltas) jako další features.

    Args:
        skeleton: (T, K, D) - only first 2 dims (xy) used

    Returns:
        (T, K, 4) where last dim = [x, y, dx, dy]
    """
    sk = skeleton[..., :2]
    velocity = np.zeros_like(sk)
    velocity[1:] = sk[1:] - sk[:-1]   # forward differences, first frame zero
    return np.concatenate([sk, velocity], axis=-1)

In [7]:
# === Cell 4: Augmentation ===
# CrowdPose left-right keypoint pairs pro mirror flip
LEFT_RIGHT_PAIRS = [
    (0, 1),    # shoulders
    (2, 3),    # elbows
    (4, 5),    # wrists
    (6, 7),    # hips
    (8, 9),    # knees
    (10, 11),  # ankles
    # head (12), neck (13) jsou centrální - neflipuje se
]


def augment_skeleton(skeleton):
    """Random flip / rotation / scale / jitter na centered skeleton."""
    sk = skeleton.copy()

    # Horizontal flip (50% chance)
    if random.random() < 0.5:
        sk[..., 0] = -sk[..., 0]
        for l, r in LEFT_RIGHT_PAIRS:
            sk[:, [l, r]] = sk[:, [r, l]]

    # Random rotation (±8 deg)
    angle = np.random.uniform(-0.14, 0.14)
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]], dtype=sk.dtype)
    sk[..., :2] = sk[..., :2] @ R.T

    # Random scale
    sk[..., :2] *= np.random.uniform(0.9, 1.1)

    # Jitter
    sk[..., :2] += np.random.normal(0, 0.02, sk[..., :2].shape).astype(sk.dtype)

    return sk

In [ ]:
# === Cell 5: Dataset class (RWF only, 2-class) ===
class SkeletonDataset(Dataset):
    """RWF skeleton dataset s sliding window samplingem. Binary labels (0=NonFight, 1=Fight)."""

    def __init__(self, rwf_dir, split='train', sequence_length=SEQUENCE_LENGTH,
                 step_size=STEP_SIZE, augment=False):
        self.sequence_length = sequence_length
        self.step_size = step_size
        self.augment = augment
        self.samples = []   # list of (skeleton_window (T,K,2), label int)

        for cls_dir, label in [('NonFight', 0), ('Fight', 1)]:
            d = os.path.join(rwf_dir, split, cls_dir)
            if not os.path.exists(d):
                print(f'⚠ {d} not found')
                continue
            for pkl_path in sorted(glob.glob(os.path.join(d, '*.pkl'))):
                try:
                    with open(pkl_path, 'rb') as f:
                        result = pickle.load(f)
                except Exception as e:
                    print(f'⚠ Skip {pkl_path}: {e}')
                    continue
                for track in result.get('person_tracks', []):
                    sk = track['skeletons'][..., :2]   # (T, 14, 2) — drop confidence
                    if len(sk) < self.sequence_length:
                        continue
                    for i in range(0, len(sk) - self.sequence_length + 1, self.step_size):
                        window = sk[i:i + self.sequence_length]
                        self.samples.append((window, label))

        if not self.samples:
            print(f'⚠ No samples loaded from {rwf_dir}/{split}!')
            return

        labels = [s[1] for s in self.samples]
        print(f'\nDataset {split}: {len(self.samples)} samples')
        for c in range(N_CLASSES):
            n = labels.count(c)
            pct = 100 * n / len(labels) if labels else 0
            print(f'  {c} ({CLASS_NAMES[c]}): {n} ({pct:.1f}%)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        skeleton, label = self.samples[idx]
        skeleton = skeleton.copy().astype(np.float32)
        skeleton = np.nan_to_num(skeleton)

        # Normalize (center+scale)
        skeleton = normalize_skeleton(skeleton)

        # Augment AFTER normalize
        if self.augment:
            skeleton = augment_skeleton(skeleton)

        # Velocity features → (T, 14, 4)
        skeleton = add_velocity_features(skeleton)

        # Flatten last 2 dims → (T, 14*4=56)
        T = skeleton.shape[0]
        skeleton = skeleton.reshape(T, -1)

        return torch.from_numpy(skeleton).float(), torch.tensor(label, dtype=torch.long)

In [9]:
# === Cell 6: Model — BiLSTM + multi-head attention ===
class BiLSTMAttention(nn.Module):
    """
    Architektura:
        input (B, T, 56) → linear projection (B, T, H)
                         → BiLSTM (B, T, 2H)
                         → Multi-head self-attention + LayerNorm + residual
                         → mean pool over time (B, 2H)
                         → MLP classifier (B, n_classes)
    """
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, n_layers=N_LSTM_LAYERS,
                 n_classes=N_CLASSES, dropout=DROPOUT, attention_heads=ATTENTION_HEADS):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,
            num_heads=attention_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        # x: (B, T, D)
        x = self.input_proj(x)              # (B, T, H)
        x, _ = self.lstm(x)                 # (B, T, 2H)
        attn_out, _ = self.attention(x, x, x)
        x = self.norm(x + attn_out)         # residual + norm
        x = x.mean(dim=1)                   # global temporal pool → (B, 2H)
        return self.classifier(x)           # (B, n_classes)

In [10]:
# === Cell 7: Loss functions ===
class FocalLoss(nn.Module):
    """Focal loss for class imbalance. gamma=0 reduces to weighted CE."""
    def __init__(self, alpha=None, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha   # tensor (n_classes,) or None
        self.gamma = gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

In [ ]:
# === Cell 8: Build datasets + dataloaders (RWF only, train+val splits) ===
print('Loading TRAIN split...')
train_dataset = SkeletonDataset(RWF_SKELETONS_DIR, split='train', augment=USE_AUGMENTATION)

print('\nLoading VAL split...')
val_dataset = SkeletonDataset(RWF_SKELETONS_DIR, split='val', augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'\nTrain loader: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val loader: {len(val_dataset)} samples, {len(val_loader)} batches')

# Class weights z trainu
labels = np.array([s[1] for s in train_dataset.samples])
class_counts = np.bincount(labels, minlength=N_CLASSES)
class_counts = np.maximum(class_counts, 1)
class_weights = len(labels) / (N_CLASSES * class_counts)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print(f'\nClass counts: {class_counts}')
print(f'Class weights: {class_weights}')

In [12]:
# === Cell 9: Build model + optimizer ===
input_dim = N_KEYPOINTS * 4   # xy + velocity_xy
model = BiLSTMAttention(input_dim=input_dim).to(DEVICE)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f'\nTrainable params: {n_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

if USE_FOCAL_LOSS:
    loss_fn = FocalLoss(alpha=class_weights_t, gamma=FOCAL_GAMMA)
    print(f'Loss: FocalLoss (gamma={FOCAL_GAMMA})')
else:
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)
    print('Loss: weighted CrossEntropyLoss')

BiLSTMAttention(
  (input_proj): Linear(in_features=56, out_features=128, bias=True)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=3, bias=True)
  )
)

Trainable params: 963,715
Loss: FocalLoss (gamma=2.0)


In [13]:
# === Cell 10: Training utilities ===
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    losses, correct, total = [], 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        optimizer.step()
        losses.append(loss.item())
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return float(np.mean(losses)), correct / max(total, 1)


@torch.no_grad()
def eval_model(model, loader, loss_fn, device):
    model.eval()
    losses, correct, total = [], 0, 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        losses.append(loss.item())
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return float(np.mean(losses)), correct / max(total, 1), all_preds, all_labels

In [14]:
# === Cell 11: SPUSTIT TRÉNINK (uncomment níže) ===
# history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
# best_val_acc = 0.0
#
# for epoch in range(EPOCHS):
#     tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
#     va_loss, va_acc, va_preds, va_labels = eval_model(model, val_loader, loss_fn, DEVICE)
#     scheduler.step()
#
#     for k, v in [('train_loss', tr_loss), ('val_loss', va_loss),
#                  ('train_acc', tr_acc),  ('val_acc', va_acc)]:
#         history[k].append(v)
#
#     if va_acc > best_val_acc:
#         best_val_acc = va_acc
#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'val_acc': va_acc,
#         }, MODEL_SAVE_PATH)
#
#     print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
#           f'train_loss={tr_loss:.4f} acc={tr_acc:.3f} | '
#           f'val_loss={va_loss:.4f} acc={va_acc:.3f}{"   ★ BEST" if va_acc == best_val_acc else ""}')
#
# print(f'\nBest val accuracy: {best_val_acc:.3f}')

print('Training cell ready. Uncomment to start training.')

Training cell ready. Uncomment to start training.


In [15]:
# === Cell 12: Vyhodnocení (po tréninku) ===
# # Plot history
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(history['train_loss'], label='train')
# axes[0].plot(history['val_loss'], label='val')
# axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
# axes[1].plot(history['train_acc'], label='train')
# axes[1].plot(history['val_acc'], label='val')
# axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
# plt.tight_layout(); plt.show()
#
# # Final eval
# # Load best checkpoint
# ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
# model.load_state_dict(ckpt['model_state_dict'])
# _, _, va_preds, va_labels = eval_model(model, val_loader, loss_fn, DEVICE)
#
# print('\nClassification Report:')
# print(classification_report(va_labels, va_preds, target_names=CLASS_NAMES, digits=3, zero_division=0))
#
# cm = confusion_matrix(va_labels, va_preds, labels=list(range(N_CLASSES)))
# plt.figure(figsize=(6, 5))
# sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
# plt.title('Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Predicted')
# plt.tight_layout(); plt.show()

print('Evaluation cell ready (run after training).')

Evaluation cell ready (run after training).


In [16]:
# === Cell 13: Sanity check současného stavu (rychlé) ===
# Otestujeme jeden batch z aktuálního dataloaderu, abychom viděli, že model FORWARD funguje.
print('Sanity check forward pass...')
model.eval()
with torch.no_grad():
    x, y = next(iter(train_loader))
    x, y = x.to(DEVICE), y.to(DEVICE)
    print(f'Input shape:  {tuple(x.shape)}')   # (B, T, 56)
    print(f'Labels shape: {tuple(y.shape)}')
    logits = model(x)
    print(f'Output shape: {tuple(logits.shape)}')
    print(f'Output sample (logits[0]): {logits[0].cpu().numpy()}')
    print(f'Predicted classes (first 5): {logits.argmax(1)[:5].cpu().numpy()}')
    print(f'True classes (first 5):      {y[:5].cpu().numpy()}')
print('\n✓ Forward pass OK, ready for training.')

Sanity check forward pass...
Input shape:  (20, 30, 56)
Labels shape: (20,)
Output shape: (20, 3)
Output sample (logits[0]): [-0.01708304 -0.16753452 -0.20533746]
Predicted classes (first 5): [0 0 1 0 0]
True classes (first 5):      [0 0 0 0 0]

✓ Forward pass OK, ready for training.
